# Document Databases — First Contact

Instead of rows and columns, data lives as JSON documents — self-contained, nested, schema-flexible. An endpoint with its last 5 alerts can live in ONE document; no joins needed. MongoDB Atlas is the cloud-hosted version: free tier, always on, globally distributed. You define no schema upfront — just insert a document and MongoDB stores it.

## What makes document stores different

- **Schema-flexible** — each document can have different fields; add a new field to one document without migrating the rest.
- **Nested data** — embed related data inside one document (alerts inside an endpoint doc) so one read gets everything, no joins.
- **Horizontal scale** — sharding is built in from day one; Mongo distributes data across nodes by a shard key automatically.
- **When to use** — varied or evolving schemas, hierarchical data, content stores, product catalogs, user profiles, event logs.

In [8]:
from pathlib import Path
import sys

# Locate _setup whether Jupyter CWD is the notebook folder or workspace root
for _candidate in [Path('_setup'), Path('Basics/Databases/_setup')]:
    if _candidate.exists():
        sys.path.insert(0, str(_candidate.resolve()))
        break

import pandas as pd
from db_connections import get_mongo_db

db = get_mongo_db()
print("Connected:", db.name)
print("Collections:", db.list_collection_names())

Connected: de_telemetry
Collections: ['endpoints', 'alerts']


In [9]:
# Load telemetry data into MongoDB from Postgres
# Each document = one endpoint or alert — flat structure mirroring relational for comparison
from db_connections import get_postgres_conn
import psycopg2.extras

pg = get_postgres_conn()
cur = pg.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# ── Endpoints (1 000 docs for speed) ─────────────────────────────────────────
cur.execute("SELECT * FROM telemetry.endpoints LIMIT 1000")
endpoint_docs = [dict(e) for e in cur.fetchall()]
for doc in endpoint_docs:
    doc['endpoint_id'] = str(doc['endpoint_id'])
    if doc.get('created_at'):
        doc['created_at'] = str(doc['created_at'])

col_ep = db["endpoints"]
col_ep.drop()
col_ep.insert_many(endpoint_docs)
print(f"Loaded {len(endpoint_docs):,} endpoint documents into MongoDB")

# ── Alerts (all 25 K) ─────────────────────────────────────────────────────────
cur.execute("SELECT * FROM telemetry.alerts")
alerts = [dict(a) for a in cur.fetchall()]
for doc in alerts:
    doc['alert_id']    = str(doc['alert_id'])
    doc['endpoint_id'] = str(doc['endpoint_id'])
    if doc.get('created_at'):
        doc['created_at']  = str(doc['created_at'])
    if doc.get('resolved_at'):
        doc['resolved_at'] = str(doc['resolved_at'])

col_al = db["alerts"]
col_al.drop()
col_al.insert_many(alerts)
print(f"Loaded {len(alerts):,} alert documents into MongoDB")

Loaded 1,000 endpoint documents into MongoDB
Loaded 25,000 alert documents into MongoDB


## 5 telemetry queries — MongoDB style

In [10]:
# Query 1 — endpoint count by datacenter
# MongoDB aggregation pipeline equivalent of: GROUP BY datacenter ORDER BY count DESC
pipeline = [
    {"$group": {"_id": "$datacenter", "endpoint_count": {"$sum": 1}}},
    {"$sort": {"endpoint_count": -1}}
]
result = list(db.endpoints.aggregate(pipeline))
df1 = pd.DataFrame(result).rename(columns={"_id": "datacenter"})
print("Endpoints by datacenter:")
print(df1.to_string(index=False))

Endpoints by datacenter:
datacenter  endpoint_count
      NYC2             253
      LON1             252
      SNG1             248
      NYC1             247


In [11]:
# Query 2 — active prod endpoints in NYC1
# MongoDB find() with filter — equivalent of WHERE datacenter='NYC1' AND environment='prod' AND status='active'
result = list(db.endpoints.find(
    {"datacenter": "NYC1", "environment": "prod", "status": "active"},
    {"_id": 0, "hostname": 1, "service_type": 1, "ip_address": 1}
).limit(10))
df2 = pd.DataFrame(result)
print(f"Active prod endpoints in NYC1 (showing up to 10): {len(df2)} rows")
print(df2.to_string(index=False))

Active prod endpoints in NYC1 (showing up to 10): 10 rows
               hostname service_type      ip_address
srv-00002.citi.internal      monitor    10.52.122.63
srv-00027.citi.internal       worker   172.22.192.95
srv-00032.citi.internal           db 192.168.204.126
srv-00045.citi.internal           db   10.82.182.142
srv-00058.citi.internal          web    172.23.93.25
srv-00082.citi.internal       worker    10.31.53.212
srv-00100.citi.internal        cache     192.168.4.1
srv-00101.citi.internal          web  192.168.153.72
srv-00107.citi.internal      monitor    192.168.26.2
srv-00128.citi.internal           db   172.18.56.177


In [12]:
# Query 3 — alert counts by severity
# GROUP BY severity equivalent
pipeline = [
    {"$group": {"_id": "$severity", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]
result = list(db.alerts.aggregate(pipeline))
df3 = pd.DataFrame(result).rename(columns={"_id": "severity"})
print("Alerts by severity:")
print(df3.to_string(index=False))

Alerts by severity:
severity  count
    high   6313
     low   6254
  medium   6248
critical   6185


In [13]:
# Query 4 — open critical alerts, most recent 10
result = list(db.alerts.find(
    {"severity": "critical", "status": "open"},
    {"_id": 0, "endpoint_id": 1, "category": 1, "message": 1, "created_at": 1}
).sort("created_at", -1).limit(10))
df4 = pd.DataFrame(result)
print("Open critical alerts (top 10):")
print(df4.to_string(index=False))

Open critical alerts (top 10):
                         endpoint_id                                                       message category                       created_at
f3607120-4fd5-4b4e-b1bd-fbdd33860b4b                            Load average exceeded 4x CPU count  network 2026-03-23 11:09:58.063609+00:00
10ca17b5-8de1-4266-8bf9-0eb6f2b0235d                Network throughput dropped below SLA threshold      cpu 2026-03-23 10:59:08.063609+00:00
902e9671-f34b-4909-9ec2-b0fd943421f5          CPU utilization exceeded 90% threshold for 5 minutes     disk 2026-03-23 10:51:38.063609+00:00
b6189682-ef92-4d04-9284-f74ca3f0d9a0                    TCP connection pool exhausted on port 5432   memory 2026-03-23 10:32:23.063609+00:00
4c3d4aa2-83b7-4dfb-834b-0b27374ba2e9                             SSL certificate expires in 7 days  network 2026-03-23 09:38:57.063609+00:00
6f2835f1-45eb-40f3-b2ce-828bc8f0141a                  Memory usage at 95% — potential OOM imminent     disk 2026-03-23 08:5

In [14]:
# Query 5 — distinct alert categories
# Equivalent of: SELECT DISTINCT category FROM telemetry.alerts
categories = db.alerts.distinct("category")
print(f"Alert categories ({len(categories)} unique):")
for c in sorted(categories):
    print(f"  {c}")

Alert categories (5 unique):
  cpu
  disk
  memory
  network
  process


## SQL vs MongoDB — same question, different syntax

| SQL | MongoDB |
|-----|---------|
| `SELECT * FROM alerts WHERE severity = 'critical'` | `db.alerts.find({"severity": "critical"})` |
| `GROUP BY datacenter` | `$group: {"_id": "$datacenter"}` |
| `ORDER BY count DESC` | `$sort: {"count": -1}` |
| `LIMIT 10` | `.limit(10)` |
| `SELECT DISTINCT category` | `db.alerts.distinct("category")` |
| `SELECT col1, col2` | `{"_id": 0, "col1": 1, "col2": 1}` (projection) |
| `WHERE a AND b` | `{"a": val, "b": val}` (implicit AND) |

## Key observations

- **Schema flexibility** — we inserted 25K alert docs with no `CREATE TABLE`, no schema definition; MongoDB accepted all of them including any field variation.
- **Aggregation pipeline** — MongoDB's GROUP BY chains stages like Unix pipes: `$match` → `$group` → `$sort` → `$project`. Each stage feeds the next.
- **No joins used** — in a real document model, alerts would be embedded inside endpoint documents so one `find()` returns everything. We kept them in separate collections here to mirror the relational layout for comparison.
- **When this wins over Postgres** — schema evolves fast (new fields added weekly), data is naturally hierarchical (order with line items with product details), or write-scale requires sharding across many nodes.
- **When Postgres still wins** — complex joins across many entities, strong ACID transactions across collections, and reporting that needs arbitrary cross-collection aggregations.